In [1]:
import json
import joblib
import pandas as pd
import numpy as np
from typing import Any
from pathlib import Path
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.features import prep_one_split, load_features, FEATURES_DIR, DROP_COLS
from credit_risk.evaluation import compute_metrics
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
    brier_score_loss
)

2026-06-21 12:07:32.644 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [2]:
train_df, val_df, test_df, _ = load_splits(path=AFTER_EDA)

2026-06-21 12:07:42.678 | INFO     | credit_risk.dataset:load_splits:263 - Checking if the files exists...
2026-06-21 12:07:42.686 | INFO     | credit_risk.dataset:load_splits:265 - Loading the Cached files...
2026-06-21 12:07:43.173 | INFO     | credit_risk.dataset:load_splits:273 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [3]:
feature_splits = load_features(path=FEATURES_DIR)

2026-06-21 12:07:53.796 | INFO     | credit_risk.features:load_features:263 - Loading the processed features...
2026-06-21 12:07:54.202 | INFO     | credit_risk.features:load_features:270 - Loaded successfully!


In [4]:
X_test = feature_splits['test'][0].to_numpy()

In [5]:
cwd = Path.cwd()
project_root = cwd.parent
model_path = project_root / 'models' / 'tuned_xgb'

In [6]:
tuned_xgb_model = joblib.load(model_path / 'model.pkl')

In [7]:
with open(model_path / 'metric.json', 'r') as f:
    metrics = json.load(f)

In [8]:
test_proba = tuned_xgb_model[1].predict_proba(X_test)[:,1]

In [9]:
filtered_test_df, f_y_test = prep_one_split(df=test_df)

2026-06-21 12:09:11.320 | INFO     | credit_risk.features:prep_one_split:217 - Inside Function: prep_one_split
2026-06-21 12:09:11.320 | INFO     | credit_risk.features:sorting_with_issue_d:143 - Sorting the dataframe wrt to issue_d...
2026-06-21 12:09:11.557 | INFO     | credit_risk.features:sorting_with_issue_d:148 - Sorted successfully!
2026-06-21 12:09:11.557 | INFO     | credit_risk.features:split_target_and_features:153 - Inside Function: split_target_and_features
2026-06-21 12:09:11.557 | INFO     | credit_risk.features:split_target_and_features:154 - Splitting the target and the features...
2026-06-21 12:09:11.558 | INFO     | credit_risk.features:split_target_and_features:157 - features and target are splitted successfully...
2026-06-21 12:09:11.558 | INFO     | credit_risk.features:add_credit_yrs:170 - Inside Function: add_credit_yrs
2026-06-21 12:09:11.558 | INFO     | credit_risk.features:add_credit_yrs:172 - Adding the feature 'credit_yrs'
2026-06-21 12:09:11.563 | INFO   

In [10]:
y_proba = pd.Series(test_proba, index=f_y_test.index)

In [26]:
def get_fairness_audit_metric(feature: pd.Series, y_proba: pd.Series, y_true: pd.Series, threshold: float) -> list[dict[str, Any]]:
    """Calculates the fairness audit metrics for each segment of a feature"""
    y_pred = pd.Series((y_proba >= threshold).astype(int), index=y_true.index)
    results = []
    segments = feature.value_counts().index.to_list()
    for segment in segments:
        idx = feature[feature == segment].index
        seg_y_true = y_true.loc[idx]
        seg_y_pred = y_pred.loc[idx]
        
        n = len(idx)
        n_pos = seg_y_true.sum()
        n_neg = n - n_pos
        
        if n_pos == 0 or n_neg == 0:
            continue
        
        # base_rate
        base_rate = seg_y_true.sum() / n
        
        # selection rate
        selection_rate = ((seg_y_pred == 0).sum()) / n
        
        # tpr
        tpr = ((seg_y_pred == 1) & (seg_y_true == 1)).sum() / n_pos
        
        # fpr
        fpr = ((seg_y_pred == 1) & (seg_y_true == 0)).sum() / n_neg
        
        # precision
        n_pred_pos = (seg_y_pred == 1).sum()
        precision = ((seg_y_pred == 1) & (seg_y_true == 1)).sum() / n_pred_pos if n_pred_pos > 0 else float('nan')
        
        results.append({
            'segment': segment,
            'n': n,
            'n_positives': n_pos,
            'base_rate': base_rate,
            'selection_rate': selection_rate,
            'tpr': tpr,
            'fpr': fpr,
            'precision': precision
        })
        
    return results

In [27]:
annual_inc_cuts = pd.cut(
    filtered_test_df['annual_inc'],
    bins=[0, 40_000, 80_000, 120_000, float('inf')],
    labels=['<$40k', '$40k-$80k', '$80k-$120k', '>$120k']
)

In [28]:
annual_inc_fair_metrics = pd.DataFrame(get_fairness_audit_metric(feature=annual_inc_cuts, y_proba=y_proba, y_true=f_y_test, threshold=metrics['threshold']))
annual_inc_fair_metrics

,segment,n,n_positives,base_rate,selection_rate,tpr,fpr,precision
0,$40k-$80k,208784,36909,0.176781,0.547834,0.678723,0.403514,0.265357
1,$80k-$120k,96273,14341,0.148962,0.661359,0.578272,0.296697,0.254371
2,<$40k,72497,14754,0.203512,0.399341,0.769622,0.557487,0.260759
3,>$120k,54101,6783,0.125377,0.724109,0.496241,0.244304,0.225513


In [29]:
home_ownership_fairness_metrics = pd.DataFrame(get_fairness_audit_metric(feature=filtered_test_df['home_ownership'], y_proba=y_proba, y_true=f_y_test, threshold=metrics['threshold']))
home_ownership_fairness_metrics

,segment,n,n_positives,base_rate,selection_rate,tpr,fpr,precision
0,MORTGAGE,210252,30118,0.143247,0.637445,0.598081,0.323176,0.236304
1,RENT,168655,33738,0.200042,0.485980,0.721205,0.462210,0.280672
2,OWN,52695,8924,0.169352,0.571477,0.640968,0.385209,0.253310
3,ANY,110,18,0.163636,0.718182,0.611111,0.217391,0.354839
